In [2]:
import pandas as pd
df = pd.read_csv("commonsenseqa_val_en_500.csv")
print(df.shape)
print(df.columns)
df.head(3)

(500, 4)
Index(['id', 'question', 'options', 'answer_key'], dtype='object')


,id,question,options,answer_key
0,c0d75f9fbf30aa3a612f16edb20d6b8d,The man took paperwork to other people to cons...,A) desk\nB) meeting\nC) office\nD) table\nE) work,B
1,c21ec5b367f409a0288d616f626555ae,"If a person is working a lot, what are they li...",A) much money\nB) own house\nC) creativity\nD)...,A
2,e6586bba9fe96d38792e6e6d4f2703dc,Where can you put a picture frame when it's no...,A) art show\nB) wall\nC) newspaper\nD) car\nE)...,E


In [3]:
import re

def parse_options(options_str: str) -> dict:
   
    opts = {}
    
    matches = re.findall(r"([ABCDE])\)\s*(.*?)(?=\n[ABCDE]\)|\Z)", options_str.strip(), flags=re.S)
    for letter, text in matches:
        opts[letter] = text.strip()
   
    if set(opts.keys()) != set(list("ABCDE")):
        raise ValueError(f"Options parsing failed. Got keys={opts.keys()} from:\n{options_str}")
    return opts

In [4]:
from openai import OpenAI

client = OpenAI(api_key=" ")

def csqa_en_basic_predict(question: str, options_str: str) -> str:
    choices = parse_options(options_str)

    prompt = f"""You are a commonsense reasoning expert.

Answer the following multiple-choice question.
Respond with only ONE letter: A, B, C, D, or E.

Question:
{question}

Choices:
A) {choices['A']}
B) {choices['B']}
C) {choices['C']}
D) {choices['D']}
E) {choices['E']}
"""
    resp = client.chat.completions.create(
        model="gpt-4o",
        temperature=0,
        messages=[{"role": "user", "content": prompt}],
    )
    out = resp.choices[0].message.content.strip().upper()
    m = re.search(r"\b([ABCDE])\b", out)
    return m.group(1) if m else out

In [5]:
for i in range(5):
    pred = csqa_en_basic_predict(df.loc[i, "question"], df.loc[i, "options"])
    print(i, "pred:", pred, "gold:", df.loc[i, "answer_key"])

0 pred: B gold: B
1 pred: A gold: A
2 pred: E gold: E
3 pred: A gold: A
4 pred: E gold: E


In [6]:
import time

if "pred_en_basic" not in df.columns:
    df["pred_en_basic"] = ""

save_path = "csqa_en_basic_progress.csv"

for i in range(len(df)):
    if str(df.loc[i, "pred_en_basic"]).strip() in list("ABCDE"):
        continue  

    print(f"CSQA EN-Basic {i+1}/{len(df)}")
    df.loc[i, "pred_en_basic"] = csqa_en_basic_predict(df.loc[i, "question"], df.loc[i, "options"])

    if (i + 1) % 25 == 0:
        df.to_csv(save_path, index=False)
        print("Saved:", save_path, "at", i+1)

    time.sleep(0.25)  

df.to_csv("csqa_en_basic_final.csv", index=False)
print("Saved: csqa_en_basic_final.csv")

CSQA EN-Basic 1/500
CSQA EN-Basic 2/500
CSQA EN-Basic 3/500
CSQA EN-Basic 4/500
CSQA EN-Basic 5/500
CSQA EN-Basic 6/500
CSQA EN-Basic 7/500
CSQA EN-Basic 8/500
CSQA EN-Basic 9/500
CSQA EN-Basic 10/500
CSQA EN-Basic 11/500
CSQA EN-Basic 12/500
CSQA EN-Basic 13/500
CSQA EN-Basic 14/500
CSQA EN-Basic 15/500
CSQA EN-Basic 16/500
CSQA EN-Basic 17/500
CSQA EN-Basic 18/500
CSQA EN-Basic 19/500
CSQA EN-Basic 20/500
CSQA EN-Basic 21/500
CSQA EN-Basic 22/500
CSQA EN-Basic 23/500
CSQA EN-Basic 24/500
CSQA EN-Basic 25/500
Saved: csqa_en_basic_progress.csv at 25
CSQA EN-Basic 26/500
CSQA EN-Basic 27/500
CSQA EN-Basic 28/500
CSQA EN-Basic 29/500
CSQA EN-Basic 30/500
CSQA EN-Basic 31/500
CSQA EN-Basic 32/500
CSQA EN-Basic 33/500
CSQA EN-Basic 34/500
CSQA EN-Basic 35/500
CSQA EN-Basic 36/500
CSQA EN-Basic 37/500
CSQA EN-Basic 38/500
CSQA EN-Basic 39/500
CSQA EN-Basic 40/500
CSQA EN-Basic 41/500
CSQA EN-Basic 42/500
CSQA EN-Basic 43/500
CSQA EN-Basic 44/500
CSQA EN-Basic 45/500
CSQA EN-Basic 46/500
CSQ

CSQA EN-Basic 353/500
CSQA EN-Basic 354/500
CSQA EN-Basic 355/500
CSQA EN-Basic 356/500
CSQA EN-Basic 357/500
CSQA EN-Basic 358/500
CSQA EN-Basic 359/500
CSQA EN-Basic 360/500
CSQA EN-Basic 361/500
CSQA EN-Basic 362/500
CSQA EN-Basic 363/500
CSQA EN-Basic 364/500
CSQA EN-Basic 365/500
CSQA EN-Basic 366/500
CSQA EN-Basic 367/500
CSQA EN-Basic 368/500
CSQA EN-Basic 369/500
CSQA EN-Basic 370/500
CSQA EN-Basic 371/500
CSQA EN-Basic 372/500
CSQA EN-Basic 373/500
CSQA EN-Basic 374/500
CSQA EN-Basic 375/500
Saved: csqa_en_basic_progress.csv at 375
CSQA EN-Basic 376/500
CSQA EN-Basic 377/500
CSQA EN-Basic 378/500
CSQA EN-Basic 379/500
CSQA EN-Basic 380/500
CSQA EN-Basic 381/500
CSQA EN-Basic 382/500
CSQA EN-Basic 383/500
CSQA EN-Basic 384/500
CSQA EN-Basic 385/500
CSQA EN-Basic 386/500
CSQA EN-Basic 387/500
CSQA EN-Basic 388/500
CSQA EN-Basic 389/500
CSQA EN-Basic 390/500
CSQA EN-Basic 391/500
CSQA EN-Basic 392/500
CSQA EN-Basic 393/500
CSQA EN-Basic 394/500
CSQA EN-Basic 395/500
CSQA EN-Basic

In [7]:
from sklearn.metrics import accuracy_score, classification_report

y_true = df["answer_key"].astype(str).str.strip().str.upper()
y_pred = df["pred_en_basic"].astype(str).str.strip().str.upper()

print("Accuracy:", accuracy_score(y_true, y_pred))
print(classification_report(y_true, y_pred, labels=list("ABCDE")))

Accuracy: 0.884
              precision    recall  f1-score   support

           A       0.89      0.92      0.91       100
           B       0.91      0.91      0.91       118
           C       0.88      0.91      0.89        87
           D       0.87      0.85      0.86       102
           E       0.87      0.83      0.85        93

    accuracy                           0.88       500
   macro avg       0.88      0.88      0.88       500
weighted avg       0.88      0.88      0.88       500



In [8]:
import pandas as pd

df = pd.read_csv("commonsenseqa_kz_500.csv")  
print(df.columns)
df.head(2)

Index(['id', 'question', 'options', 'answer_key', 'question_kz', 'options_kz'], dtype='object')


,id,question,options,answer_key,question_kz,options_kz
0,c0d75f9fbf30aa3a612f16edb20d6b8d,The man took paperwork to other people to cons...,A) desk\nB) meeting\nC) office\nD) table\nE) work,B,Ер адам құжаттарды басқа адамдармен кеңесу үші...,A) үстел\nB) жиналыс\nC) кеңсе\nD) үстел\nE) ж...
1,c21ec5b367f409a0288d616f626555ae,"If a person is working a lot, what are they li...",A) much money\nB) own house\nC) creativity\nD)...,A,"Егер адам көп жұмыс істесе, олар нені табуға т...",A) көп ақша\nB) өз үйі\nC) шығармашылық\nD) жа...


In [9]:
def csqa_native_basic_kz_predict(question_kz: str, options_kz: str) -> str:
    prompt = f"""Сен күнделікті өмірге қатысты логикалық (commonsense) сұрақтарды жақсы түсінетін сарапшысың.

Төмендегі көп таңдаулы сұраққа дұрыс жауапты таңда.
Тек бір әріппен жауап бер: A, B, C, D немесе E.
Ешқандай түсіндірме қоспа.

СҰРАҚ:
{question_kz}

ЖАУАП НҰСҚАЛАРЫ:
{options_kz}
"""
    resp = client.chat.completions.create(
        model="gpt-4o",
        temperature=0,
        messages=[{"role": "user", "content": prompt}],
    )
    out = resp.choices[0].message.content.strip().upper()
    m = re.search(r"\b([ABCDE])\b", out)
    return m.group(1) if m else out

In [10]:
for i in range(5):
    pred = csqa_native_basic_kz_predict(df.loc[i, "question_kz"], df.loc[i, "options_kz"])
    print(i, "pred:", pred, "gold:", df.loc[i, "answer_key"])
    time.sleep(0.25)

0 pred: B gold: B
1 pred: A gold: A
2 pred: B gold: E
3 pred: A gold: A
4 pred: E gold: E


In [11]:
if "pred_native_kz" not in df.columns:
    df["pred_native_kz"] = ""

progress_path = "csqa_pred_native_kz_progress.csv"
final_path = "csqa_pred_native_kz_final.csv"

for i in range(len(df)):
    if str(df.loc[i, "pred_native_kz"]).strip() in list("ABCDE"):
        continue  # resume-safe

    print(f"CSQA Native-Basic KZ {i+1}/{len(df)}")
    df.loc[i, "pred_native_kz"] = csqa_native_basic_kz_predict(
        df.loc[i, "question_kz"],
        df.loc[i, "options_kz"],
    )

    if (i + 1) % 25 == 0:
        df.to_csv(progress_path, index=False)
        print("Saved:", progress_path)

    time.sleep(0.25)

df.to_csv(final_path, index=False)
print("Saved:", final_path)

CSQA Native-Basic KZ 1/500
CSQA Native-Basic KZ 2/500
CSQA Native-Basic KZ 3/500
CSQA Native-Basic KZ 4/500
CSQA Native-Basic KZ 5/500
CSQA Native-Basic KZ 6/500
CSQA Native-Basic KZ 7/500
CSQA Native-Basic KZ 8/500
CSQA Native-Basic KZ 9/500
CSQA Native-Basic KZ 10/500
CSQA Native-Basic KZ 11/500
CSQA Native-Basic KZ 12/500
CSQA Native-Basic KZ 13/500
CSQA Native-Basic KZ 14/500
CSQA Native-Basic KZ 15/500
CSQA Native-Basic KZ 16/500
CSQA Native-Basic KZ 17/500
CSQA Native-Basic KZ 18/500
CSQA Native-Basic KZ 19/500
CSQA Native-Basic KZ 20/500
CSQA Native-Basic KZ 21/500
CSQA Native-Basic KZ 22/500
CSQA Native-Basic KZ 23/500
CSQA Native-Basic KZ 24/500
CSQA Native-Basic KZ 25/500
Saved: csqa_pred_native_kz_progress.csv
CSQA Native-Basic KZ 26/500
CSQA Native-Basic KZ 27/500
CSQA Native-Basic KZ 28/500
CSQA Native-Basic KZ 29/500
CSQA Native-Basic KZ 30/500
CSQA Native-Basic KZ 31/500
CSQA Native-Basic KZ 32/500
CSQA Native-Basic KZ 33/500
CSQA Native-Basic KZ 34/500
CSQA Native-Basic

CSQA Native-Basic KZ 274/500
CSQA Native-Basic KZ 275/500
Saved: csqa_pred_native_kz_progress.csv
CSQA Native-Basic KZ 276/500
CSQA Native-Basic KZ 277/500
CSQA Native-Basic KZ 278/500
CSQA Native-Basic KZ 279/500
CSQA Native-Basic KZ 280/500
CSQA Native-Basic KZ 281/500
CSQA Native-Basic KZ 282/500
CSQA Native-Basic KZ 283/500
CSQA Native-Basic KZ 284/500
CSQA Native-Basic KZ 285/500
CSQA Native-Basic KZ 286/500
CSQA Native-Basic KZ 287/500
CSQA Native-Basic KZ 288/500
CSQA Native-Basic KZ 289/500
CSQA Native-Basic KZ 290/500
CSQA Native-Basic KZ 291/500
CSQA Native-Basic KZ 292/500
CSQA Native-Basic KZ 293/500
CSQA Native-Basic KZ 294/500
CSQA Native-Basic KZ 295/500
CSQA Native-Basic KZ 296/500
CSQA Native-Basic KZ 297/500
CSQA Native-Basic KZ 298/500
CSQA Native-Basic KZ 299/500
CSQA Native-Basic KZ 300/500
Saved: csqa_pred_native_kz_progress.csv
CSQA Native-Basic KZ 301/500
CSQA Native-Basic KZ 302/500
CSQA Native-Basic KZ 303/500
CSQA Native-Basic KZ 304/500
CSQA Native-Basic KZ 

In [12]:
df_eval = pd.read_csv(final_path)

y_true = df_eval["answer_key"].astype(str).str.strip().str.upper()
y_pred = df_eval["pred_native_kz"].astype(str).str.strip().str.upper()

print("Accuracy:", accuracy_score(y_true, y_pred))
print(classification_report(y_true, y_pred, labels=list("ABCDE")))

Accuracy: 0.738
              precision    recall  f1-score   support

           A       0.74      0.80      0.77       100
           B       0.76      0.76      0.76       118
           C       0.67      0.64      0.66        87
           D       0.74      0.73      0.73       102
           E       0.76      0.74      0.75        93

    accuracy                           0.74       500
   macro avg       0.74      0.73      0.73       500
weighted avg       0.74      0.74      0.74       500



In [13]:
def csqa_en_prompt_kz_input(question_kz, options_kz):

    prompt = f"""You are a commonsense reasoning expert.

Answer the following multiple choice question.

Respond with ONLY one letter: A, B, C, D, or E.
Do not explain your answer.

Question:
{question_kz}

Choices:
{options_kz}
"""

    response = client.chat.completions.create(
        model="gpt-4o",
        temperature=0,
        messages=[{"role":"user","content":prompt}]
    )

    output = response.choices[0].message.content.strip().upper()

    match = re.search(r"[ABCDE]", output)
    return match.group(0) if match else output

In [14]:
for i in range(5):

    pred = csqa_en_prompt_kz_input(
        df.loc[i,"question_kz"],
        df.loc[i,"options_kz"]
    )

    print(i, "pred:", pred, "gold:", df.loc[i,"answer_key"])

0 pred: B gold: B
1 pred: A gold: A
2 pred: E gold: E
3 pred: A gold: A
4 pred: E gold: E


In [15]:
import time

df["pred_en_kz"] = ""

for i in range(len(df)):

    print(f"EN prompt + KZ input {i+1}/500")

    df.loc[i,"pred_en_kz"] = csqa_en_prompt_kz_input(
        df.loc[i,"question_kz"],
        df.loc[i,"options_kz"]
    )

    if (i+1) % 25 == 0:
        df.to_csv("csqa_en_kz_progress.csv", index=False)

    time.sleep(0.25)

df.to_csv("csqa_en_kz_final.csv", index=False)

print("Finished EN prompt + KZ input experiment")

EN prompt + KZ input 1/500
EN prompt + KZ input 2/500
EN prompt + KZ input 3/500
EN prompt + KZ input 4/500
EN prompt + KZ input 5/500
EN prompt + KZ input 6/500
EN prompt + KZ input 7/500
EN prompt + KZ input 8/500
EN prompt + KZ input 9/500
EN prompt + KZ input 10/500
EN prompt + KZ input 11/500
EN prompt + KZ input 12/500
EN prompt + KZ input 13/500
EN prompt + KZ input 14/500
EN prompt + KZ input 15/500
EN prompt + KZ input 16/500
EN prompt + KZ input 17/500
EN prompt + KZ input 18/500
EN prompt + KZ input 19/500
EN prompt + KZ input 20/500
EN prompt + KZ input 21/500
EN prompt + KZ input 22/500
EN prompt + KZ input 23/500
EN prompt + KZ input 24/500
EN prompt + KZ input 25/500
EN prompt + KZ input 26/500
EN prompt + KZ input 27/500
EN prompt + KZ input 28/500
EN prompt + KZ input 29/500
EN prompt + KZ input 30/500
EN prompt + KZ input 31/500
EN prompt + KZ input 32/500
EN prompt + KZ input 33/500
EN prompt + KZ input 34/500
EN prompt + KZ input 35/500
EN prompt + KZ input 36/500
E

EN prompt + KZ input 288/500
EN prompt + KZ input 289/500
EN prompt + KZ input 290/500
EN prompt + KZ input 291/500
EN prompt + KZ input 292/500
EN prompt + KZ input 293/500
EN prompt + KZ input 294/500
EN prompt + KZ input 295/500
EN prompt + KZ input 296/500
EN prompt + KZ input 297/500
EN prompt + KZ input 298/500
EN prompt + KZ input 299/500
EN prompt + KZ input 300/500
EN prompt + KZ input 301/500
EN prompt + KZ input 302/500
EN prompt + KZ input 303/500
EN prompt + KZ input 304/500
EN prompt + KZ input 305/500
EN prompt + KZ input 306/500
EN prompt + KZ input 307/500
EN prompt + KZ input 308/500
EN prompt + KZ input 309/500
EN prompt + KZ input 310/500
EN prompt + KZ input 311/500
EN prompt + KZ input 312/500
EN prompt + KZ input 313/500
EN prompt + KZ input 314/500
EN prompt + KZ input 315/500
EN prompt + KZ input 316/500
EN prompt + KZ input 317/500
EN prompt + KZ input 318/500
EN prompt + KZ input 319/500
EN prompt + KZ input 320/500
EN prompt + KZ input 321/500
EN prompt + KZ

In [16]:
y_true = df["answer_key"].str.strip().str.upper()
y_pred = df["pred_en_kz"].str.strip().str.upper()

print("Accuracy:", accuracy_score(y_true, y_pred))
print(classification_report(y_true, y_pred))

Accuracy: 0.758
              precision    recall  f1-score   support

           A       0.79      0.84      0.81       100
           B       0.80      0.76      0.78       118
           C       0.71      0.69      0.70        87
           D       0.77      0.74      0.75       102
           E       0.71      0.75      0.73        93

    accuracy                           0.76       500
   macro avg       0.76      0.76      0.76       500
weighted avg       0.76      0.76      0.76       500



In [ ]:
def csqa_cot_kz_predict(question_kz: str, options_kz: str) -> str:
    prompt = f"""Сен commonsense логика бойынша сарапшысың.

Төмендегі көп таңдаулы сұраққа жауап бер.
Алдымен қысқаша (1–2 сөйлем) ойыңды жаз, содан кейін соңғы жауапты бер.

Формат (міндетті):
Ой: <қысқаша reasoning>
Жауап: <A/B/C/D/E>

СҰРАҚ:
{question_kz}

НҰСҚАЛАР:
{options_kz}
"""
    resp = client.chat.completions.create(
        model="gpt-4o",
        temperature=0,
        messages=[{"role": "user", "content": prompt}],
    )
    out = resp.choices[0].message.content.strip().upper()

    # Prefer "ЖАУАП: X" / "ANSWER: X"
    m = re.search(r"(ЖАУАП|ANSWER)\s*:\s*([ABCDE])", out)
    if m:
        return m.group(2)

    # fallback: first standalone letter
    m2 = re.search(r"\b([ABCDE])\b", out)
    return m2.group(1) if m2 else out

In [19]:
def csqa_cot_kz(question_kz, options_kz):

    prompt = f"""Сен күнделікті өмірдегі логикалық сұрақтарды жақсы түсінетін сарапшысың.

Төмендегі көп таңдаулы сұраққа жауап бер.

Алдымен қысқаша ойыңды жаз (1–2 сөйлем).
Содан кейін соңғы жауапты жаңа жолда дәл мына форматта жаз:

Жауап: <A/B/C/D/E>

Сұрақ:
{question_kz}

Жауап нұсқалары:
{options_kz}
"""

    response = client.chat.completions.create(
        model="gpt-4o",
        temperature=0,
        messages=[{"role":"user","content":prompt}]
    )

    output = response.choices[0].message.content.strip().upper()

    match = re.search(r"ЖАУАП\s*:\s*([ABCDE])", output)

    if match:
        return match.group(1)

    # fallback
    match2 = re.search(r"[ABCDE]", output)
    return match2.group(0) if match2 else output

In [18]:
for i in range(5):

    pred = csqa_cot_kz(
        df.loc[i,"question_kz"],
        df.loc[i,"options_kz"]
    )

    print(i, "pred:", pred, "gold:", df.loc[i,"answer_key"])

0 pred: B gold: B
1 pred: A gold: A
2 pred: E gold: E
3 pred: A gold: A
4 pred: E gold: E


In [20]:
import time

df["pred_cot_kz"] = ""

for i in range(len(df)):

    print(f"Kazakh CoT {i+1}/500")

    df.loc[i,"pred_cot_kz"] = csqa_cot_kz(
        df.loc[i,"question_kz"],
        df.loc[i,"options_kz"]
    )

    if (i+1) % 25 == 0:
        df.to_csv("csqa_cot_kz_progress.csv", index=False)

    time.sleep(0.25)

df.to_csv("csqa_cot_kz_final.csv", index=False)

print("Kazakh CoT experiment finished")

Kazakh CoT 1/500
Kazakh CoT 2/500
Kazakh CoT 3/500
Kazakh CoT 4/500
Kazakh CoT 5/500
Kazakh CoT 6/500
Kazakh CoT 7/500
Kazakh CoT 8/500
Kazakh CoT 9/500
Kazakh CoT 10/500
Kazakh CoT 11/500
Kazakh CoT 12/500
Kazakh CoT 13/500
Kazakh CoT 14/500
Kazakh CoT 15/500
Kazakh CoT 16/500
Kazakh CoT 17/500
Kazakh CoT 18/500
Kazakh CoT 19/500
Kazakh CoT 20/500
Kazakh CoT 21/500
Kazakh CoT 22/500
Kazakh CoT 23/500
Kazakh CoT 24/500
Kazakh CoT 25/500
Kazakh CoT 26/500
Kazakh CoT 27/500
Kazakh CoT 28/500
Kazakh CoT 29/500
Kazakh CoT 30/500
Kazakh CoT 31/500
Kazakh CoT 32/500
Kazakh CoT 33/500
Kazakh CoT 34/500
Kazakh CoT 35/500
Kazakh CoT 36/500
Kazakh CoT 37/500
Kazakh CoT 38/500
Kazakh CoT 39/500
Kazakh CoT 40/500
Kazakh CoT 41/500
Kazakh CoT 42/500
Kazakh CoT 43/500
Kazakh CoT 44/500
Kazakh CoT 45/500
Kazakh CoT 46/500
Kazakh CoT 47/500
Kazakh CoT 48/500
Kazakh CoT 49/500
Kazakh CoT 50/500
Kazakh CoT 51/500
Kazakh CoT 52/500
Kazakh CoT 53/500
Kazakh CoT 54/500
Kazakh CoT 55/500
Kazakh CoT 56/500
K

Kazakh CoT 438/500
Kazakh CoT 439/500
Kazakh CoT 440/500
Kazakh CoT 441/500
Kazakh CoT 442/500
Kazakh CoT 443/500
Kazakh CoT 444/500
Kazakh CoT 445/500
Kazakh CoT 446/500
Kazakh CoT 447/500
Kazakh CoT 448/500
Kazakh CoT 449/500
Kazakh CoT 450/500
Kazakh CoT 451/500
Kazakh CoT 452/500
Kazakh CoT 453/500
Kazakh CoT 454/500
Kazakh CoT 455/500
Kazakh CoT 456/500
Kazakh CoT 457/500
Kazakh CoT 458/500
Kazakh CoT 459/500
Kazakh CoT 460/500
Kazakh CoT 461/500
Kazakh CoT 462/500
Kazakh CoT 463/500
Kazakh CoT 464/500
Kazakh CoT 465/500
Kazakh CoT 466/500
Kazakh CoT 467/500
Kazakh CoT 468/500
Kazakh CoT 469/500
Kazakh CoT 470/500
Kazakh CoT 471/500
Kazakh CoT 472/500
Kazakh CoT 473/500
Kazakh CoT 474/500
Kazakh CoT 475/500
Kazakh CoT 476/500
Kazakh CoT 477/500
Kazakh CoT 478/500
Kazakh CoT 479/500
Kazakh CoT 480/500
Kazakh CoT 481/500
Kazakh CoT 482/500
Kazakh CoT 483/500
Kazakh CoT 484/500
Kazakh CoT 485/500
Kazakh CoT 486/500
Kazakh CoT 487/500
Kazakh CoT 488/500
Kazakh CoT 489/500
Kazakh CoT 4

In [21]:
y_true = df["answer_key"].str.strip().str.upper()
y_pred = df["pred_cot_kz"].str.strip().str.upper()

print("Accuracy:", accuracy_score(y_true, y_pred))
print(classification_report(y_true, y_pred))

Accuracy: 0.722
              precision    recall  f1-score   support

           A       0.75      0.78      0.76       100
           B       0.75      0.73      0.74       118
           C       0.67      0.61      0.64        87
           D       0.71      0.77      0.74       102
           E       0.72      0.70      0.71        93

    accuracy                           0.72       500
   macro avg       0.72      0.72      0.72       500
weighted avg       0.72      0.72      0.72       500



In [22]:
def csqa_cot_en(question, options):

    prompt = f"""You are a commonsense reasoning expert.

Solve the multiple-choice question below.

First think step by step about the problem.
Then give the final answer on a new line in this exact format:

Answer: <A/B/C/D/E>

Question:
{question}

Choices:
{options}
"""

    response = client.chat.completions.create(
        model="gpt-4o",
        temperature=0,
        messages=[{"role":"user","content":prompt}]
    )

    output = response.choices[0].message.content.strip().upper()

    match = re.search(r"ANSWER\s*:\s*([ABCDE])", output)

    if match:
        return match.group(1)

    # fallback
    match2 = re.search(r"[ABCDE]", output)
    return match2.group(0) if match2 else output

In [23]:
for i in range(5):

    pred = csqa_cot_en(
        df.loc[i,"question"],
        df.loc[i,"options"]
    )

    print(i, "pred:", pred, "gold:", df.loc[i,"answer_key"])

0 pred: B gold: B
1 pred: A gold: A
2 pred: E gold: E
3 pred: A gold: A
4 pred: E gold: E


In [24]:
df["pred_cot_en"] = ""

for i in range(len(df)):

    print(f"English CoT {i+1}/500")

    df.loc[i,"pred_cot_en"] = csqa_cot_en(
        df.loc[i,"question"],
        df.loc[i,"options"]
    )

    if (i+1) % 25 == 0:
        df.to_csv("csqa_cot_en_progress.csv", index=False)

    time.sleep(0.25)

df.to_csv("csqa_cot_en_final.csv", index=False)

print("English CoT experiment finished")

English CoT 1/500
English CoT 2/500
English CoT 3/500
English CoT 4/500
English CoT 5/500
English CoT 6/500
English CoT 7/500
English CoT 8/500
English CoT 9/500
English CoT 10/500
English CoT 11/500
English CoT 12/500
English CoT 13/500
English CoT 14/500
English CoT 15/500
English CoT 16/500
English CoT 17/500
English CoT 18/500
English CoT 19/500
English CoT 20/500
English CoT 21/500
English CoT 22/500
English CoT 23/500
English CoT 24/500
English CoT 25/500
English CoT 26/500
English CoT 27/500
English CoT 28/500
English CoT 29/500
English CoT 30/500
English CoT 31/500
English CoT 32/500
English CoT 33/500
English CoT 34/500
English CoT 35/500
English CoT 36/500
English CoT 37/500
English CoT 38/500
English CoT 39/500
English CoT 40/500
English CoT 41/500
English CoT 42/500
English CoT 43/500
English CoT 44/500
English CoT 45/500
English CoT 46/500
English CoT 47/500
English CoT 48/500
English CoT 49/500
English CoT 50/500
English CoT 51/500
English CoT 52/500
English CoT 53/500
En

English CoT 417/500
English CoT 418/500
English CoT 419/500
English CoT 420/500
English CoT 421/500
English CoT 422/500
English CoT 423/500
English CoT 424/500
English CoT 425/500
English CoT 426/500
English CoT 427/500
English CoT 428/500
English CoT 429/500
English CoT 430/500
English CoT 431/500
English CoT 432/500
English CoT 433/500
English CoT 434/500
English CoT 435/500
English CoT 436/500
English CoT 437/500
English CoT 438/500
English CoT 439/500
English CoT 440/500
English CoT 441/500
English CoT 442/500
English CoT 443/500
English CoT 444/500
English CoT 445/500
English CoT 446/500
English CoT 447/500
English CoT 448/500
English CoT 449/500
English CoT 450/500
English CoT 451/500
English CoT 452/500
English CoT 453/500
English CoT 454/500
English CoT 455/500
English CoT 456/500
English CoT 457/500
English CoT 458/500
English CoT 459/500
English CoT 460/500
English CoT 461/500
English CoT 462/500
English CoT 463/500
English CoT 464/500
English CoT 465/500
English CoT 466/500


In [25]:
y_true = df["answer_key"].str.strip().str.upper()
y_pred = df["pred_cot_en"].str.strip().str.upper()

print("Accuracy:", accuracy_score(y_true, y_pred))
print(classification_report(y_true, y_pred))

Accuracy: 0.876
              precision    recall  f1-score   support

           A       0.92      0.89      0.90       100
           B       0.93      0.87      0.90       118
           C       0.84      0.92      0.88        87
           D       0.85      0.84      0.85       102
           E       0.83      0.86      0.85        93

    accuracy                           0.88       500
   macro avg       0.87      0.88      0.88       500
weighted avg       0.88      0.88      0.88       500



In [26]:
def csqa_cot_en_kz(question_kz, options_kz):

    prompt = f"""You are a commonsense reasoning expert.

Think step by step before answering the question.

After reasoning, provide the final answer in the following format:

Answer: <A/B/C/D/E>

Question:
{question_kz}

Choices:
{options_kz}
"""

    response = client.chat.completions.create(
        model="gpt-4o",
        temperature=0,
        messages=[{"role":"user","content":prompt}]
    )

    output = response.choices[0].message.content.strip().upper()

    match = re.search(r"ANSWER\s*:\s*([ABCDE])", output)

    if match:
        return match.group(1)

    match2 = re.search(r"[ABCDE]", output)
    return match2.group(0) if match2 else output

In [27]:
for i in range(5):

    pred = csqa_cot_en_kz(
        df.loc[i,"question_kz"],
        df.loc[i,"options_kz"]
    )

    print(i, "pred:", pred, "gold:", df.loc[i,"answer_key"])

0 pred: B gold: B
1 pred: A gold: A
2 pred: E gold: E
3 pred: A gold: A
4 pred: E gold: E


In [28]:
import time

df["pred_cot_en_kz"] = ""

for i in range(len(df)):

    print(f"EN CoT + KZ input {i+1}/500")

    df.loc[i,"pred_cot_en_kz"] = csqa_cot_en_kz(
        df.loc[i,"question_kz"],
        df.loc[i,"options_kz"]
    )

    if (i+1) % 25 == 0:
        df.to_csv("csqa_cot_en_kz_progress.csv", index=False)

    time.sleep(0.25)

df.to_csv("csqa_cot_en_kz_final.csv", index=False)

print("EN CoT + KZ experiment finished")

EN CoT + KZ input 1/500
EN CoT + KZ input 2/500
EN CoT + KZ input 3/500
EN CoT + KZ input 4/500
EN CoT + KZ input 5/500
EN CoT + KZ input 6/500
EN CoT + KZ input 7/500
EN CoT + KZ input 8/500
EN CoT + KZ input 9/500
EN CoT + KZ input 10/500
EN CoT + KZ input 11/500
EN CoT + KZ input 12/500
EN CoT + KZ input 13/500
EN CoT + KZ input 14/500
EN CoT + KZ input 15/500
EN CoT + KZ input 16/500
EN CoT + KZ input 17/500
EN CoT + KZ input 18/500
EN CoT + KZ input 19/500
EN CoT + KZ input 20/500
EN CoT + KZ input 21/500
EN CoT + KZ input 22/500
EN CoT + KZ input 23/500
EN CoT + KZ input 24/500
EN CoT + KZ input 25/500
EN CoT + KZ input 26/500
EN CoT + KZ input 27/500
EN CoT + KZ input 28/500
EN CoT + KZ input 29/500
EN CoT + KZ input 30/500
EN CoT + KZ input 31/500
EN CoT + KZ input 32/500
EN CoT + KZ input 33/500
EN CoT + KZ input 34/500
EN CoT + KZ input 35/500
EN CoT + KZ input 36/500
EN CoT + KZ input 37/500
EN CoT + KZ input 38/500
EN CoT + KZ input 39/500
EN CoT + KZ input 40/500
EN CoT + 

EN CoT + KZ input 321/500
EN CoT + KZ input 322/500
EN CoT + KZ input 323/500
EN CoT + KZ input 324/500
EN CoT + KZ input 325/500
EN CoT + KZ input 326/500
EN CoT + KZ input 327/500
EN CoT + KZ input 328/500
EN CoT + KZ input 329/500
EN CoT + KZ input 330/500
EN CoT + KZ input 331/500
EN CoT + KZ input 332/500
EN CoT + KZ input 333/500
EN CoT + KZ input 334/500
EN CoT + KZ input 335/500
EN CoT + KZ input 336/500
EN CoT + KZ input 337/500
EN CoT + KZ input 338/500
EN CoT + KZ input 339/500
EN CoT + KZ input 340/500
EN CoT + KZ input 341/500
EN CoT + KZ input 342/500
EN CoT + KZ input 343/500
EN CoT + KZ input 344/500
EN CoT + KZ input 345/500
EN CoT + KZ input 346/500
EN CoT + KZ input 347/500
EN CoT + KZ input 348/500
EN CoT + KZ input 349/500
EN CoT + KZ input 350/500
EN CoT + KZ input 351/500
EN CoT + KZ input 352/500
EN CoT + KZ input 353/500
EN CoT + KZ input 354/500
EN CoT + KZ input 355/500
EN CoT + KZ input 356/500
EN CoT + KZ input 357/500
EN CoT + KZ input 358/500
EN CoT + KZ 

In [29]:
y_true = df["answer_key"].str.strip().str.upper()
y_pred = df["pred_cot_en_kz"].str.strip().str.upper()

print("Accuracy:", accuracy_score(y_true, y_pred))
print(classification_report(y_true, y_pred))

Accuracy: 0.744
              precision    recall  f1-score   support

           A       0.75      0.81      0.78       100
           B       0.79      0.75      0.77       118
           C       0.69      0.68      0.68        87
           D       0.74      0.76      0.75       102
           E       0.73      0.71      0.72        93

    accuracy                           0.74       500
   macro avg       0.74      0.74      0.74       500
weighted avg       0.74      0.74      0.74       500

